# 🎓 CampusPilot AI - AI-Powered College Information Assistant

## Phase 1 — Dataset Generation

This notebook generates a complete synthetic College ERP database for the CampusPilot AI project.

The generated datasets are used later for:
- Retrieval-Augmented Generation (RAG)
- Agentic AI
- Streamlit-based College Assistant

The AI implementation is developed separately in the VS Code project.


## 📖 Project Overview



CampusPilot AI is an AI-powered college information assistant that uses
Retrieval-Augmented Generation (RAG) to answer queries related to students,
faculty, departments, attendance, courses, and academic records.

This project simulates a college ERP system by generating realistic relational
datasets and uses Google's Gemini API with LangChain and FAISS to provide
intelligent responses.

# 📑 Table of Contents



## 1. PROJECT SETUP

### A. Install Required Libraries

In [136]:
!pip -q install \
langchain \
langchain-community \
langchain-google-genai \
google-genai \
faiss-cpu \
sentence-transformers \
pypdf \
faker

### B. Import Libraries


In [137]:
import os
import pandas as pd
import numpy as np
import random

from faker import Faker
from google import genai
from datetime import datetime, timedelta

In [138]:
# Verify Faker Installation
fake = Faker("en_IN")

print(fake.name())
print(fake.email())
print(fake.phone_number())

Diya Mangal
aarna96@example.org
08651562131


### C. Project Setup

In [139]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [140]:
# Create Project Folder Structure
PROJECT_PATH = "/content/drive/MyDrive/CampusPilotAI"

folders = [
    f"{PROJECT_PATH}/data",
    f"{PROJECT_PATH}/docs",
    f"{PROJECT_PATH}/vector_db",
    f"{PROJECT_PATH}/assets"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created successfully!")

✅ Project folders created successfully!


## 2. CONFIGURATION

### A. College Configuration

In [141]:
COLLEGE_NAME = "CampusPilot Institute of Technology"

COLLEGE_SHORT_NAME = "CPIT"

EMAIL_DOMAIN = "cpit.edu.in"

ACADEMIC_YEAR = "2026-27"

STUDENTS_PER_SECTION = 60

SECTION_NAMES = ["A", "B", "C"]

PROGRAMS = {

    "B.Tech": {

        "duration": 4,

        "departments": {

            "CSE": {
                "name": "Computer Science Engineering",
                "sections_per_year": 3,
                "total_faculty": 24,
                "office": "Academic Block A - 201"
            },

            "CSEDS": {
                "name": "Computer Science & Data Science",
                "sections_per_year": 1,
                "total_faculty": 10,
                "office": "Academic Block A - 202"
            },

            "IT": {
                "name": "Information Technology",
                "sections_per_year": 3,
                "total_faculty": 22,
                "office": "Academic Block A - 203"
            },

            "ECE": {
                "name": "Electronics & Communication Engineering",
                "sections_per_year": 2,
                "total_faculty": 16,
                "office": "Academic Block B - 101"
            },

            "EEE": {
                "name": "Electrical & Electronics Engineering",
                "sections_per_year": 1,
                "total_faculty": 10,
                "office": "Academic Block B - 102"
            }
        }
    },

    "BBA": {

        "duration": 3,

        "departments": {

            "BBA": {
                "name": "Bachelor of Business Administration",
                "sections_per_year": 2,
                "total_faculty": 12,
                "office": "Management Block - 201"
            }
        }
    },

    "MBA": {

        "duration": 2,

        "departments": {

            "MBA": {
                "name": "Master of Business Administration",
                "sections_per_year": 2,
                "total_faculty": 10,
                "office": "Management Block - 202"
            }
        }
    }
}


### B. Global Configuration

In [142]:
GENDERS = ["Male", "Female"]

### C. Global Helper Functions

In [143]:
def generate_id(prefix, number, digits=3):
    """
    Generate IDs like:
    FAC001
    STU0001
    SUB012
    """
    return f"{prefix}{number:0{digits}d}"

def generate_name():
    """
    Generate a realistic Indian first and last name.
    """
    first_name = fake.first_name()
    last_name = fake.last_name()

    return first_name, last_name

def generate_email(first_name, last_name):
    """
    Generate official college email.
    Example:
    rahul.sharma@cpit.edu.in
    """
    first = first_name.lower().replace(" ", "")
    last = last_name.lower().replace(" ", "")

    return f"{first}.{last}@{EMAIL_DOMAIN}"


def generate_phone():
    """
    Generate a realistic Indian mobile number.
    """
    return random.choice(["9", "8", "7", "6"]) + "".join(
        random.choices("0123456789", k=9)
    )

def generate_roll_number(department, admission_year, section, sequence):
    """
    Example:
    CSE2026A001
    IT2025B023
    MBA2026A014
    """
    return f"{department}{admission_year}{section}{sequence:03d}"


## 3. DATASET GENERATION

### A. Departments Dataset

#### Generate Dataset

In [144]:
departments = []

for program, program_info in PROGRAMS.items():

    duration = program_info["duration"]

    for dept_id, dept_info in program_info["departments"].items():

        total_sections = dept_info["sections_per_year"] * duration

        total_students = total_sections * STUDENTS_PER_SECTION

        departments.append({

            "Department_ID": dept_id,

            "Department_Name": dept_info["name"],

            "Program": program,

            "Duration_Years": duration,

            "Sections_Per_Year": dept_info["sections_per_year"],

            "Total_Sections": total_sections,

            "Students_Per_Section": STUDENTS_PER_SECTION,

            "Total_Students": total_students,

            "Total_Faculty": dept_info["total_faculty"],

            "HOD_ID": "",          # Will be assigned after faculty generation

            "Office": dept_info["office"],

            "Email": f"{dept_id.lower()}@{EMAIL_DOMAIN}"

        })

departments_df = pd.DataFrame(departments)

departments_df

,Department_ID,Department_Name,Program,Duration_Years,Sections_Per_Year,Total_Sections,Students_Per_Section,Total_Students,Total_Faculty,HOD_ID,Office,Email
0,CSE,Computer Science Engineering,B.Tech,4,3,12,60,720,24,,Academic Block A - 201,cse@cpit.edu.in
1,CSEDS,Computer Science & Data Science,B.Tech,4,1,4,60,240,10,,Academic Block A - 202,cseds@cpit.edu.in
2,IT,Information Technology,B.Tech,4,3,12,60,720,22,,Academic Block A - 203,it@cpit.edu.in
3,ECE,Electronics & Communication Engineering,B.Tech,4,2,8,60,480,16,,Academic Block B - 101,ece@cpit.edu.in
4,EEE,Electrical & Electronics Engineering,B.Tech,4,1,4,60,240,10,,Academic Block B - 102,eee@cpit.edu.in
5,BBA,Bachelor of Business Administration,BBA,3,2,6,60,360,12,,Management Block - 201,bba@cpit.edu.in
6,MBA,Master of Business Administration,MBA,2,2,4,60,240,10,,Management Block - 202,mba@cpit.edu.in


#### Save Dataset

In [145]:
departments_df.to_csv(
    f"{PROJECT_PATH}/data/departments.csv",
    index=False
)

print("departments.csv saved successfully!")

departments.csv saved successfully!


#### Verify Dataset

In [146]:
print("College Name:", COLLEGE_NAME)

print("Total Departments:", len(departments_df))

print("Total Sections:", departments_df["Total_Sections"].sum())

print("Total Students:", departments_df["Total_Students"].sum())

print("Total Faculty:", departments_df["Total_Faculty"].sum())

College Name: CampusPilot Institute of Technology
Total Departments: 7
Total Sections: 50
Total Students: 3000
Total Faculty: 104


In [147]:
print("Shape:", departments_df.shape)

print("\nColumns:")
print(departments_df.columns.tolist())

print("\nMissing Values:")
print(departments_df.isnull().sum())

departments_df.head()

Shape: (7, 12)

Columns:
['Department_ID', 'Department_Name', 'Program', 'Duration_Years', 'Sections_Per_Year', 'Total_Sections', 'Students_Per_Section', 'Total_Students', 'Total_Faculty', 'HOD_ID', 'Office', 'Email']

Missing Values:
Department_ID           0
Department_Name         0
Program                 0
Duration_Years          0
Sections_Per_Year       0
Total_Sections          0
Students_Per_Section    0
Total_Students          0
Total_Faculty           0
HOD_ID                  0
Office                  0
Email                   0
dtype: int64


,Department_ID,Department_Name,Program,Duration_Years,Sections_Per_Year,Total_Sections,Students_Per_Section,Total_Students,Total_Faculty,HOD_ID,Office,Email
0,CSE,Computer Science Engineering,B.Tech,4,3,12,60,720,24,,Academic Block A - 201,cse@cpit.edu.in
1,CSEDS,Computer Science & Data Science,B.Tech,4,1,4,60,240,10,,Academic Block A - 202,cseds@cpit.edu.in
2,IT,Information Technology,B.Tech,4,3,12,60,720,22,,Academic Block A - 203,it@cpit.edu.in
3,ECE,Electronics & Communication Engineering,B.Tech,4,2,8,60,480,16,,Academic Block B - 101,ece@cpit.edu.in
4,EEE,Electrical & Electronics Engineering,B.Tech,4,1,4,60,240,10,,Academic Block B - 102,eee@cpit.edu.in


### B. Faculty Dataset

#### Configuration

In [148]:
DESIGNATIONS = {
    "Professor": 0.10,
    "Associate Professor": 0.20,
    "Assistant Professor": 0.70
}

QUALIFICATIONS = {
    "B.Tech": ["M.Tech", "PhD"],
    "BBA": ["MBA", "PhD"],
    "MBA": ["MBA", "PhD"]
}

#### Generate Dataset

In [149]:
faculty = []

faculty_counter = 1

current_year = int(ACADEMIC_YEAR[:4])

for _, dept in departments_df.iterrows():

    for _ in range(dept["Total_Faculty"]):

        # Name
        first_name, last_name = generate_name()
        full_name = f"{first_name} {last_name}"

        # Gender
        gender = random.choice(GENDERS)

        # Designation
        designation = random.choices(
            population=list(DESIGNATIONS.keys()),
            weights=list(DESIGNATIONS.values()),
            k=1
        )[0]

        # Qualification
        qualification = random.choice(
            QUALIFICATIONS[dept["Program"]]
        )

        # Experience
        experience = random.randint(1, 30)

        # Joining Year
        joining_year = current_year - experience

        faculty.append({

            "Faculty_ID": generate_id("FAC", faculty_counter),

            "First_Name": first_name,

            "Last_Name": last_name,

            "Full_Name": full_name,

            "Gender": gender,

            "Department_ID": dept["Department_ID"],

            "Program": dept["Program"],

            "Designation": designation,

            "Qualification": qualification,

            "Experience_Years": experience,

            "Joining_Year": joining_year,

            "Email": generate_email(first_name, last_name),

            "Phone": generate_phone(),

            "Is_HOD": "No"

        })

        faculty_counter += 1

faculty_df = pd.DataFrame(faculty)

faculty_df

,Faculty_ID,First_Name,Last_Name,Full_Name,Gender,Department_ID,Program,Designation,Qualification,Experience_Years,Joining_Year,Email,Phone,Is_HOD
0,FAC001,Suhani,Rajagopal,Suhani Rajagopal,Female,CSE,B.Tech,Assistant Professor,M.Tech,21,2005,suhani.rajagopal@cpit.edu.in,6371398984,No
1,FAC002,Anita,Contractor,Anita Contractor,Female,CSE,B.Tech,Associate Professor,PhD,9,2017,anita.contractor@cpit.edu.in,8691235938,No
2,FAC003,Zayan,Venkatesh,Zayan Venkatesh,Male,CSE,B.Tech,Associate Professor,PhD,25,2001,zayan.venkatesh@cpit.edu.in,9921968804,No
3,FAC004,Nitesh,Rai,Nitesh Rai,Female,CSE,B.Tech,Assistant Professor,M.Tech,10,2016,nitesh.rai@cpit.edu.in,9232523780,No
4,FAC005,Madhavi,Pant,Madhavi Pant,Female,CSE,B.Tech,Assistant Professor,M.Tech,7,2019,madhavi.pant@cpit.edu.in,6124807877,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,FAC100,Nihal,Krishnamurthy,Nihal Krishnamurthy,Male,MBA,MBA,Associate Professor,PhD,7,2019,nihal.krishnamurthy@cpit.edu.in,8052149025,No
100,FAC101,Bachittar,Sahni,Bachittar Sahni,Male,MBA,MBA,Associate Professor,PhD,9,2017,bachittar.sahni@cpit.edu.in,9624795758,No
101,FAC102,Urishilla,Srinivasan,Urishilla Srinivasan,Female,MBA,MBA,Assistant Professor,PhD,22,2004,urishilla.srinivasan@cpit.edu.in,8373600973,No
102,FAC103,Saanvi,Parmar,Saanvi Parmar,Male,MBA,MBA,Assistant Professor,MBA,21,2005,saanvi.parmar@cpit.edu.in,6092574629,No


#### Update Department HODs

In [150]:
for dept_id in faculty_df["Department_ID"].unique():

    # Faculty belonging to this department
    dept_faculty = faculty_df[faculty_df["Department_ID"] == dept_id]

    # Prefer Professors, then Associate Professors, then anyone
    professors = dept_faculty[
        dept_faculty["Designation"] == "Professor"
    ]

    associates = dept_faculty[
        dept_faculty["Designation"] == "Associate Professor"
    ]

    if not professors.empty:
        hod = professors.sample(1).index[0]

    elif not associates.empty:
        hod = associates.sample(1).index[0]

    else:
        hod = dept_faculty.sample(1).index[0]

    # Update faculty dataset
    faculty_df.loc[hod, "Is_HOD"] = "Yes"

    # Update departments dataset
    departments_df.loc[
        departments_df["Department_ID"] == dept_id,
        "HOD_ID"
    ] = faculty_df.loc[hod, "Faculty_ID"]

print("HODs assigned successfully!")

HODs assigned successfully!


In [151]:
#verification: one HOD per department
faculty_df[faculty_df["Is_HOD"] == "Yes"][
    ["Faculty_ID", "Full_Name", "Department_ID", "Designation"]
]

,Faculty_ID,Full_Name,Department_ID,Designation
7,FAC008,Nathaniel Bhasin,CSE,Professor
26,FAC027,Indira Sabharwal,CSEDS,Professor
38,FAC039,Pranit Ghosh,IT,Professor
68,FAC069,Matthew Bains,ECE,Professor
74,FAC075,Owen Shankar,EEE,Professor
84,FAC085,Nitara Mukhopadhyay,BBA,Professor
98,FAC099,Rohan Mishra,MBA,Professor


In [152]:
departments_df[
    ["Department_ID", "HOD_ID"]
]

,Department_ID,HOD_ID
0,CSE,FAC008
1,CSEDS,FAC027
2,IT,FAC039
3,ECE,FAC069
4,EEE,FAC075
5,BBA,FAC085
6,MBA,FAC099


#### Save both datasets

In [153]:
faculty_df.to_csv(
    f"{PROJECT_PATH}/data/faculty.csv",
    index=False
)

departments_df.to_csv(
    f"{PROJECT_PATH}/data/departments.csv",
    index=False
)

print("faculty.csv saved successfully!")
print("departments.csv updated successfully!")

faculty.csv saved successfully!
departments.csv updated successfully!


### C. Sections Dataset

#### Generate Dataset

In [154]:
sections = []

section_counter = 1

for _, dept in departments_df.iterrows():

    department = dept["Department_ID"]
    program = dept["Program"]

    duration = dept["Duration_Years"]
    sections_per_year = dept["Sections_Per_Year"]

    # Faculty in this department
    dept_faculty = faculty_df[
        faculty_df["Department_ID"] == department
    ]

    # Prefer Assistant Professors
    advisor_pool = dept_faculty[
        dept_faculty["Designation"] == "Assistant Professor"
    ]

    # If not enough, include Associate Professors
    if len(advisor_pool) < sections_per_year * duration:
        advisor_pool = dept_faculty[
            dept_faculty["Designation"].isin(
                ["Assistant Professor", "Associate Professor"]
            )
        ]

    # If still not enough, use everyone
    if len(advisor_pool) < sections_per_year * duration:
        advisor_pool = dept_faculty

    advisor_pool = advisor_pool["Faculty_ID"].tolist()

    faculty_index = 0

    for year in range(1, duration + 1):

        for section in SECTION_NAMES[:sections_per_year]:

            advisor = advisor_pool[faculty_index]

            sections.append({

                "Section_ID": generate_id("SEC", section_counter),

                "Section_Name": f"{department}-{year}{section}",

                "Department_ID": department,

                "Program": program,

                "Year": year,

                "Section": section,

                "Faculty_Advisor_ID": advisor,

                "Class_Strength": STUDENTS_PER_SECTION

            })

            section_counter += 1

            # Move to next faculty (round robin)
            faculty_index = (faculty_index + 1) % len(dept_faculty)

sections_df = pd.DataFrame(sections)

sections_df

,Section_ID,Section_Name,Department_ID,Program,Year,Section,Faculty_Advisor_ID,Class_Strength
0,SEC001,CSE-1A,CSE,B.Tech,1,A,FAC001,60
1,SEC002,CSE-1B,CSE,B.Tech,1,B,FAC004,60
2,SEC003,CSE-1C,CSE,B.Tech,1,C,FAC005,60
3,SEC004,CSE-2A,CSE,B.Tech,2,A,FAC006,60
4,SEC005,CSE-2B,CSE,B.Tech,2,B,FAC007,60
5,SEC006,CSE-2C,CSE,B.Tech,2,C,FAC009,60
6,SEC007,CSE-3A,CSE,B.Tech,3,A,FAC010,60
7,SEC008,CSE-3B,CSE,B.Tech,3,B,FAC011,60
8,SEC009,CSE-3C,CSE,B.Tech,3,C,FAC013,60
9,SEC010,CSE-4A,CSE,B.Tech,4,A,FAC015,60


#### Save Dataset

In [155]:
sections_df.to_csv(
    f"{PROJECT_PATH}/data/sections.csv",
    index=False
)

print("sections.csv saved successfully!")

sections.csv saved successfully!


#### Verify Dataset

In [156]:
print("Total Sections:", len(sections_df))

sections_df.head(10)

Total Sections: 50


,Section_ID,Section_Name,Department_ID,Program,Year,Section,Faculty_Advisor_ID,Class_Strength
0,SEC001,CSE-1A,CSE,B.Tech,1,A,FAC001,60
1,SEC002,CSE-1B,CSE,B.Tech,1,B,FAC004,60
2,SEC003,CSE-1C,CSE,B.Tech,1,C,FAC005,60
3,SEC004,CSE-2A,CSE,B.Tech,2,A,FAC006,60
4,SEC005,CSE-2B,CSE,B.Tech,2,B,FAC007,60
5,SEC006,CSE-2C,CSE,B.Tech,2,C,FAC009,60
6,SEC007,CSE-3A,CSE,B.Tech,3,A,FAC010,60
7,SEC008,CSE-3B,CSE,B.Tech,3,B,FAC011,60
8,SEC009,CSE-3C,CSE,B.Tech,3,C,FAC013,60
9,SEC010,CSE-4A,CSE,B.Tech,4,A,FAC015,60


### D. Students Dataset

#### Configuration

In [157]:
BLOOD_GROUPS = [
    "A+", "A-", "B+", "B-",
    "AB+", "AB-", "O+", "O-"
]

HOSTEL_STATUS = [
    "Hosteller",
    "Day Scholar"
]

#### Generate Dataset

In [158]:
students = []

student_counter = 1

current_year = int(ACADEMIC_YEAR[:4])

# Keeps track of roll number sequence within each section
section_roll_counter = {}

for _, section in sections_df.iterrows():

    section_id = section["Section_ID"]
    department = section["Department_ID"]
    program = section["Program"]
    year = section["Year"]
    section_name = section["Section"]

    # Admission year
    admission_year = current_year - (year - 1)

    # Start roll numbers from 1 for every section
    section_roll_counter[section_id] = 1

    for _ in range(STUDENTS_PER_SECTION):

        first_name, last_name = generate_name()
        full_name = f"{first_name} {last_name}"

        gender = random.choice(GENDERS)

        roll_number = generate_roll_number(
            department,
            admission_year,
            section_name,
            section_roll_counter[section_id]
        )

        students.append({

            "Student_ID": generate_id("STU", student_counter, 4),

            "Roll_Number": roll_number,

            "First_Name": first_name,

            "Last_Name": last_name,

            "Full_Name": full_name,

            "Gender": gender,

            "Department_ID": department,

            "Program": program,

            "Year": year,

            "Section_ID": section_id,

            "Email": generate_email(first_name, last_name),

            "Phone": generate_phone(),

            "Blood_Group": random.choice(BLOOD_GROUPS),

            "Hostel_Status": random.choice(HOSTEL_STATUS),

            "Admission_Year": admission_year,

            "CGPA": round(random.uniform(6.0, 10.0), 2)

        })

        student_counter += 1
        section_roll_counter[section_id] += 1

students_df = pd.DataFrame(students)

students_df

,Student_ID,Roll_Number,First_Name,Last_Name,Full_Name,Gender,Department_ID,Program,Year,Section_ID,Email,Phone,Blood_Group,Hostel_Status,Admission_Year,CGPA
0,STU0001,CSE2026A001,Leena,Issac,Leena Issac,Male,CSE,B.Tech,1,SEC001,leena.issac@cpit.edu.in,9281487162,A-,Day Scholar,2026,7.27
1,STU0002,CSE2026A002,Rushil,Walla,Rushil Walla,Female,CSE,B.Tech,1,SEC001,rushil.walla@cpit.edu.in,7428524744,AB-,Day Scholar,2026,9.60
2,STU0003,CSE2026A003,Yagnesh,Chanda,Yagnesh Chanda,Female,CSE,B.Tech,1,SEC001,yagnesh.chanda@cpit.edu.in,6244517518,AB+,Hosteller,2026,7.85
3,STU0004,CSE2026A004,Harshil,Sethi,Harshil Sethi,Female,CSE,B.Tech,1,SEC001,harshil.sethi@cpit.edu.in,6160536004,AB-,Day Scholar,2026,9.14
4,STU0005,CSE2026A005,Rishi,Ben,Rishi Ben,Female,CSE,B.Tech,1,SEC001,rishi.ben@cpit.edu.in,9785799413,AB-,Hosteller,2026,7.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,STU2996,MBA2025B056,Quincy,Patil,Quincy Patil,Male,MBA,MBA,2,SEC050,quincy.patil@cpit.edu.in,7412951168,AB-,Hosteller,2025,7.84
2996,STU2997,MBA2025B057,Vasana,Kulkarni,Vasana Kulkarni,Female,MBA,MBA,2,SEC050,vasana.kulkarni@cpit.edu.in,8077208627,O+,Hosteller,2025,7.54
2997,STU2998,MBA2025B058,Charles,Devan,Charles Devan,Male,MBA,MBA,2,SEC050,charles.devan@cpit.edu.in,8409537183,AB+,Day Scholar,2025,6.67
2998,STU2999,MBA2025B059,Faraj,Naik,Faraj Naik,Male,MBA,MBA,2,SEC050,faraj.naik@cpit.edu.in,9973137148,AB+,Hosteller,2025,6.48


#### Save Dataset

In [159]:
students_df.to_csv(
    f"{PROJECT_PATH}/data/students.csv",
    index=False
)

print("students.csv saved successfully!")

students.csv saved successfully!


#### Verify Dataset

In [160]:
print("Total Students:", len(students_df))

students_df.head()

Total Students: 3000


,Student_ID,Roll_Number,First_Name,Last_Name,Full_Name,Gender,Department_ID,Program,Year,Section_ID,Email,Phone,Blood_Group,Hostel_Status,Admission_Year,CGPA
0,STU0001,CSE2026A001,Leena,Issac,Leena Issac,Male,CSE,B.Tech,1,SEC001,leena.issac@cpit.edu.in,9281487162,A-,Day Scholar,2026,7.27
1,STU0002,CSE2026A002,Rushil,Walla,Rushil Walla,Female,CSE,B.Tech,1,SEC001,rushil.walla@cpit.edu.in,7428524744,AB-,Day Scholar,2026,9.60
2,STU0003,CSE2026A003,Yagnesh,Chanda,Yagnesh Chanda,Female,CSE,B.Tech,1,SEC001,yagnesh.chanda@cpit.edu.in,6244517518,AB+,Hosteller,2026,7.85
3,STU0004,CSE2026A004,Harshil,Sethi,Harshil Sethi,Female,CSE,B.Tech,1,SEC001,harshil.sethi@cpit.edu.in,6160536004,AB-,Day Scholar,2026,9.14
4,STU0005,CSE2026A005,Rishi,Ben,Rishi Ben,Female,CSE,B.Tech,1,SEC001,rishi.ben@cpit.edu.in,9785799413,AB-,Hosteller,2026,7.83


### E. Subjects Dataset

#### Configuration

In [161]:
SUBJECTS = {

    "B.Tech": {

        1: {
            1: [
                ("Mathematics-I", 4),
                ("Physics", 4),
                ("Programming for Problem Solving", 4),
                ("Basic Electrical Engineering", 3),
                ("English Communication", 2)
            ],

            2: [
                ("Mathematics-II", 4),
                ("Data Structures", 4),
                ("Digital Electronics", 4),
                ("Environmental Studies", 2),
                ("Python Programming", 4)
            ]
        },

        2: {

            3: [
                ("DBMS",4),
                ("Operating Systems",4),
                ("Computer Networks",4),
                ("Discrete Mathematics",4),
                ("Java Programming",4)
            ],

            4: [
                ("Software Engineering",4),
                ("Design & Analysis of Algorithms",4),
                ("Web Technologies",4),
                ("Microprocessors",4),
                ("Probability & Statistics",3)
            ]
        },

        3: {

            5: [
                ("Machine Learning",4),
                ("Compiler Design",4),
                ("Computer Graphics",4),
                ("Cloud Computing",4),
                ("Cyber Security",3)
            ],

            6: [
                ("Artificial Intelligence",4),
                ("Big Data Analytics",4),
                ("Mobile Computing",3),
                ("IoT",3),
                ("Mini Project",2)
            ]
        },

        4: {

            7: [
                ("Deep Learning",4),
                ("Blockchain",3),
                ("DevOps",3),
                ("Professional Elective-I",4),
                ("Professional Elective-II",4)
            ],

            8: [
                ("Major Project",8),
                ("Internship",4),
                ("Professional Elective-III",4),
                ("Professional Elective-IV",4),
                ("Industrial Training",2)
            ]
        }

    },

    "BBA": {

        1:{
            1:[
                ("Business Communication",4),
                ("Financial Accounting",4),
                ("Business Economics",4),
                ("Business Mathematics",4),
                ("Computer Fundamentals",3)
            ],
            2:[
                ("Marketing Management",4),
                ("Organizational Behaviour",4),
                ("Business Law",4),
                ("Business Statistics",4),
                ("Environmental Studies",2)
            ]
        },

        2:{
            3:[
                ("Human Resource Management",4),
                ("Financial Management",4),
                ("Operations Management",4),
                ("Entrepreneurship",3),
                ("Research Methodology",3)
            ],
            4:[
                ("Consumer Behaviour",4),
                ("International Business",4),
                ("Business Ethics",3),
                ("E-Commerce",3),
                ("Summer Internship",2)
            ]
        },

        3:{
            5:[
                ("Strategic Management",4),
                ("Supply Chain Management",4),
                ("Business Analytics",4),
                ("Elective-I",3),
                ("Elective-II",3)
            ],
            6:[
                ("Project Management",4),
                ("Major Project",8),
                ("Elective-III",3),
                ("Elective-IV",3),
                ("Viva Voce",2)
            ]
        }

    },

    "MBA":{

        1:{
            1:[
                ("Managerial Economics",4),
                ("Accounting for Managers",4),
                ("Organizational Behaviour",4),
                ("Marketing Management",4),
                ("Business Statistics",3)
            ],
            2:[
                ("Financial Management",4),
                ("Operations Management",4),
                ("HR Management",4),
                ("Business Analytics",4),
                ("Business Communication",3)
            ]
        },

        2:{
            3:[
                ("Strategic Management",4),
                ("International Business",4),
                ("Elective-I",4),
                ("Elective-II",4),
                ("Research Project",2)
            ],
            4:[
                ("Major Project",8),
                ("Internship",4),
                ("Elective-III",4),
                ("Elective-IV",4),
                ("Seminar",2)
            ]
        }

    }

}

#### Generate Dataset

In [162]:
subjects = []

subject_counter = 1

for program, years in SUBJECTS.items():

    for year, semesters in years.items():

        for semester, subject_list in semesters.items():

            for subject_name, credits in subject_list:

                subjects.append({

                    "Subject_ID": generate_id("SUB", subject_counter),

                    "Subject_Name": subject_name,

                    "Program": program,

                    "Year": year,

                    "Semester": semester,

                    "Credits": credits

                })

                subject_counter += 1

subjects_df = pd.DataFrame(subjects)

subjects_df

,Subject_ID,Subject_Name,Program,Year,Semester,Credits
0,SUB001,Mathematics-I,B.Tech,1,1,4
1,SUB002,Physics,B.Tech,1,1,4
2,SUB003,Programming for Problem Solving,B.Tech,1,1,4
3,SUB004,Basic Electrical Engineering,B.Tech,1,1,3
4,SUB005,English Communication,B.Tech,1,1,2
...,...,...,...,...,...,...
85,SUB086,Major Project,MBA,2,4,8
86,SUB087,Internship,MBA,2,4,4
87,SUB088,Elective-III,MBA,2,4,4
88,SUB089,Elective-IV,MBA,2,4,4


#### Save Dataset

In [163]:
subjects_df.to_csv(
    f"{PROJECT_PATH}/data/subjects.csv",
    index=False
)

print("subjects.csv saved successfully!")

subjects.csv saved successfully!


#### Verify Dataset

In [164]:
print("Total Subjects:", len(subjects_df))

subjects_df.head()

Total Subjects: 90


,Subject_ID,Subject_Name,Program,Year,Semester,Credits
0,SUB001,Mathematics-I,B.Tech,1,1,4
1,SUB002,Physics,B.Tech,1,1,4
2,SUB003,Programming for Problem Solving,B.Tech,1,1,4
3,SUB004,Basic Electrical Engineering,B.Tech,1,1,3
4,SUB005,English Communication,B.Tech,1,1,2


### F. Courses Dataset

#### Generate Dataset

In [165]:
courses = []

course_counter = 1

for _, dept in departments_df.iterrows():

    department_id = dept["Department_ID"]

    program = dept["Program"]

    duration = dept["Duration_Years"]

    # Faculty belonging to this department
    dept_faculty = faculty_df[
        faculty_df["Department_ID"] == department_id
    ]["Faculty_ID"].tolist()

    faculty_index = 0

    # Generate courses year-wise
    for year in range(1, duration + 1):

        # Subjects of this program & year
        year_subjects = subjects_df[
            (subjects_df["Program"] == program) &
            (subjects_df["Year"] == year)
        ]

        # Sections of this department & year
        year_sections = sections_df[
            (sections_df["Department_ID"] == department_id) &
            (sections_df["Year"] == year)
        ]

        # Assign ONE faculty to each subject
        subject_faculty = {}

        for _, subject in year_subjects.iterrows():

            subject_faculty[subject["Subject_ID"]] = \
                dept_faculty[faculty_index % len(dept_faculty)]

            faculty_index += 1

        # Create one course per section per subject
        for _, section in year_sections.iterrows():

            for _, subject in year_subjects.iterrows():

                courses.append({

                    "Course_ID":
                        generate_id("CRS", course_counter),

                    "Section_ID":
                        section["Section_ID"],

                    "Subject_ID":
                        subject["Subject_ID"],

                    "Faculty_ID":
                        subject_faculty[
                            subject["Subject_ID"]
                        ]

                })

                course_counter += 1

courses_df = pd.DataFrame(courses)

courses_df

,Course_ID,Section_ID,Subject_ID,Faculty_ID
0,CRS001,SEC001,SUB001,FAC001
1,CRS002,SEC001,SUB002,FAC002
2,CRS003,SEC001,SUB003,FAC003
3,CRS004,SEC001,SUB004,FAC004
4,CRS005,SEC001,SUB005,FAC005
...,...,...,...,...
495,CRS496,SEC050,SUB086,FAC100
496,CRS497,SEC050,SUB087,FAC101
497,CRS498,SEC050,SUB088,FAC102
498,CRS499,SEC050,SUB089,FAC103


#### Save Dataset

In [166]:
courses_df.to_csv(
    f"{PROJECT_PATH}/data/courses.csv",
    index=False
)

print("courses.csv saved successfully!")

courses.csv saved successfully!


#### Verify Dataset

In [167]:
print("Total Courses:", len(courses_df))

courses_df.head()

Total Courses: 500


,Course_ID,Section_ID,Subject_ID,Faculty_ID
0,CRS001,SEC001,SUB001,FAC001
1,CRS002,SEC001,SUB002,FAC002
2,CRS003,SEC001,SUB003,FAC003
3,CRS004,SEC001,SUB004,FAC004
4,CRS005,SEC001,SUB005,FAC005


### G. Classroom Dataset

#### Configuration

In [168]:
CLASSROOMS = {

    "Academic Block A": [
        "A101","A102","A103","A104","A105",
        "A106","A107","A108","A109","A110",
        "A201","A202","A203","A204","A205",
        "A206","A207","A208","A209","A210"
    ],

    "Academic Block B": [
        "B101","B102","B103","B104","B105",
        "B106","B107","B108","B109","B110",
        "B201","B202","B203","B204","B205",
        "B206","B207","B208","B209","B210"
    ],

    "Management Block": [
        "M101","M102","M103","M104","M105",
        "M201","M202","M203","M204","M205"
    ]
}

#### Generate Dataset

In [169]:
classrooms = []

room_counter = 1

for building, rooms in CLASSROOMS.items():

    for room in rooms:

        classrooms.append({

            "Room_ID": generate_id("ROOM", room_counter),

            "Room_Number": room,

            "Building": building,

            "Capacity": 60

        })

        room_counter += 1

classrooms_df = pd.DataFrame(classrooms)

classrooms_df

,Room_ID,Room_Number,Building,Capacity
0,ROOM001,A101,Academic Block A,60
1,ROOM002,A102,Academic Block A,60
2,ROOM003,A103,Academic Block A,60
3,ROOM004,A104,Academic Block A,60
4,ROOM005,A105,Academic Block A,60
5,ROOM006,A106,Academic Block A,60
6,ROOM007,A107,Academic Block A,60
7,ROOM008,A108,Academic Block A,60
8,ROOM009,A109,Academic Block A,60
9,ROOM010,A110,Academic Block A,60


#### Save Dataset

In [170]:
classrooms_df.to_csv(
    f"{PROJECT_PATH}/data/classrooms.csv",
    index=False
)

print("classrooms.csv saved successfully!")

classrooms.csv saved successfully!


#### Assign homerooms

In [171]:
classrooms_only = classrooms_df.reset_index(drop=True)

sections_sorted = sections_df.sort_values(
    ["Department_ID", "Year", "Section"]
).reset_index(drop=True)

section_room_map = {}

for i, section in sections_sorted.iterrows():

    section_room_map[section["Section_ID"]] = classrooms_only.loc[i, "Room_ID"]

### H. Timetable Dataset
    

#### Configuration

In [172]:
TIMETABLE_SLOTS = [
    (1, "09:00-10:00"),
    (2, "10:00-11:00"),
    (3, "11:15-12:15"),
    (4, "12:15-01:15"),
    (5, "02:00-03:00"),
    (6, "03:00-04:00")
]

WORKING_DAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

#### Generate Dataset

In [173]:
timetable = []

timetable_counter = 1

for _, section in sections_df.iterrows():

    section_courses = courses_df[
        courses_df["Section_ID"] == section["Section_ID"]
    ]

    lecture_pool = []

    # Every subject appears 5 times/week
    for _, course in section_courses.iterrows():

        lecture_pool.extend([course["Course_ID"]] * 5)

    random.shuffle(lecture_pool)

    lecture_index = 0

    for day in WORKING_DAYS:

        for period, slot in TIMETABLE_SLOTS:

            # Keep 5 free periods every week
            if lecture_index >= len(lecture_pool):
                continue

            course = courses_df[
                courses_df["Course_ID"] == lecture_pool[lecture_index]
            ].iloc[0]

            timetable.append({

                "Timetable_ID": generate_id(
                    "TT",
                    timetable_counter
                ),

                "Course_ID": course["Course_ID"],

                "Subject_ID": course["Subject_ID"],

                "Section_ID": course["Section_ID"],

                "Faculty_ID": course["Faculty_ID"],

                "Room_ID": section_room_map[
                    section["Section_ID"]
                ],

                "Day": day,

                "Period": period,

                "Time_Slot": slot

            })

            lecture_index += 1

            timetable_counter += 1

timetable_df = pd.DataFrame(timetable)

timetable_df

,Timetable_ID,Course_ID,Subject_ID,Section_ID,Faculty_ID,Room_ID,Day,Period,Time_Slot
0,TT001,CRS010,SUB010,SEC001,FAC010,ROOM007,Monday,1,09:00-10:00
1,TT002,CRS003,SUB003,SEC001,FAC003,ROOM007,Monday,2,10:00-11:00
2,TT003,CRS001,SUB001,SEC001,FAC001,ROOM007,Monday,3,11:15-12:15
3,TT004,CRS005,SUB005,SEC001,FAC005,ROOM007,Monday,4,12:15-01:15
4,TT005,CRS006,SUB006,SEC001,FAC006,ROOM007,Monday,5,02:00-03:00
...,...,...,...,...,...,...,...,...,...
1495,TT1496,CRS492,SUB082,SEC050,FAC096,ROOM050,Friday,2,10:00-11:00
1496,TT1497,CRS498,SUB088,SEC050,FAC102,ROOM050,Friday,3,11:15-12:15
1497,TT1498,CRS491,SUB081,SEC050,FAC095,ROOM050,Friday,4,12:15-01:15
1498,TT1499,CRS493,SUB083,SEC050,FAC097,ROOM050,Friday,5,02:00-03:00


#### Save Dataset

In [174]:
timetable_df.to_csv(
    f"{PROJECT_PATH}/data/timetable.csv",
    index=False
)

print("timetable.csv saved successfully!")

timetable.csv saved successfully!


#### Verify Dataset

In [175]:
print("Total Timetable Entries:", len(timetable_df))

timetable_df.head()

Total Timetable Entries: 1500


,Timetable_ID,Course_ID,Subject_ID,Section_ID,Faculty_ID,Room_ID,Day,Period,Time_Slot
0,TT001,CRS010,SUB010,SEC001,FAC010,ROOM007,Monday,1,09:00-10:00
1,TT002,CRS003,SUB003,SEC001,FAC003,ROOM007,Monday,2,10:00-11:00
2,TT003,CRS001,SUB001,SEC001,FAC001,ROOM007,Monday,3,11:15-12:15
3,TT004,CRS005,SUB005,SEC001,FAC005,ROOM007,Monday,4,12:15-01:15
4,TT005,CRS006,SUB006,SEC001,FAC006,ROOM007,Monday,5,02:00-03:00


### I. Books Dataset

#### Configuration

In [176]:
NO_BOOK_SUBJECTS = {

    "Major Project",
    "Mini Project",
    "Research Project",
    "Industrial Training",
    "Internship",
    "Summer Internship",
    "Seminar",
    "Viva Voce",
    "Elective-I",
    "Elective-II",
    "Elective-III",
    "Elective-IV",
    "Professional Elective-I",
    "Professional Elective-II",
    "Professional Elective-III",
    "Professional Elective-IV"

}

BOOK_TEMPLATES = [

    "{subject}",

    "Introduction to {subject}",

    "Fundamentals of {subject}",

    "{subject} Concepts",

    "{subject} Essentials",

    "Advanced {subject}"

]

PUBLISHERS = [

    "Pearson",
    "McGraw Hill",
    "Oxford University Press",
    "Springer",
    "Wiley",
    "O'Reilly",
    "Packt"

]

#### Helper Functions

In [177]:
def generate_isbn():
    return fake.isbn13(separator="-")

def generate_shelf():
    return f"S-{random.randint(1,20):02d}-{random.randint(1,8)}"

#### Generate Dataset

In [178]:
books = []

book_counter = 1

for _, subject in subjects_df.iterrows():

    subject_name = subject["Subject_Name"]

    # Skip projects, internships etc.
    if subject_name in NO_BOOK_SUBJECTS:
        continue

    # Randomly choose 4 unique title templates
    templates = random.sample(BOOK_TEMPLATES, 4)

    for template in templates:

        title = template.format(subject=subject_name)

        total = random.randint(3, 8)

        available = random.randint(0, total)

        if available == 0:
            status = "Out of Stock"

        elif available <= 2:
            status = "Limited"

        else:
            status = "Available"

        books.append({

            "Book_ID":
                generate_id("BK", book_counter),

            "Subject_ID":
                subject["Subject_ID"],

            "Subject_Name":
                subject_name,

            "Book_Title":
                title,

            "Author":
                fake.name(),

            "Publisher":
                random.choice(PUBLISHERS),

            "ISBN":
                generate_isbn(),

            "Edition":
                f"{random.randint(1,5)} Edition",

            "Published_Year":
                random.randint(2015, 2025),

            "Total_Copies":
                total,

            "Available_Copies":
                available,

            "Shelf_No":
                generate_shelf(),

            "Status":
                status

        })

        book_counter += 1

books_df = pd.DataFrame(books)

books_df

,Book_ID,Subject_ID,Subject_Name,Book_Title,Author,Publisher,ISBN,Edition,Published_Year,Total_Copies,Available_Copies,Shelf_No,Status
0,BK001,SUB001,Mathematics-I,Mathematics-I Essentials,Christopher Raj,McGraw Hill,978-1-09-764367-7,5 Edition,2017,3,0,S-15-4,Out of Stock
1,BK002,SUB001,Mathematics-I,Advanced Mathematics-I,Yashawini Jhaveri,Packt,978-1-919993-32-4,5 Edition,2018,7,6,S-17-6,Available
2,BK003,SUB001,Mathematics-I,Mathematics-I Concepts,Neel Natt,Oxford University Press,978-1-997876-02-1,1 Edition,2025,3,1,S-03-5,Limited
3,BK004,SUB001,Mathematics-I,Introduction to Mathematics-I,Gautami Som,O'Reilly,978-1-70204-295-6,2 Edition,2022,7,6,S-13-4,Available
4,BK005,SUB002,Physics,Physics,Tripti Chand,Springer,978-1-78978-861-7,5 Edition,2016,8,0,S-06-7,Out of Stock
...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,BK264,SUB081,Strategic Management,Introduction to Strategic Management,Osha Jaggi,O'Reilly,978-1-372-01935-7,2 Edition,2015,5,2,S-09-7,Limited
264,BK265,SUB082,International Business,International Business,Qadim Ravi,Packt,978-0-604-06673-2,2 Edition,2016,8,5,S-03-4,Available
265,BK266,SUB082,International Business,Fundamentals of International Business,Mitesh Ranganathan,McGraw Hill,978-0-06-979991-6,3 Edition,2016,4,1,S-09-1,Limited
266,BK267,SUB082,International Business,Advanced International Business,Dalbir De,McGraw Hill,978-1-355-56609-0,4 Edition,2019,4,0,S-20-6,Out of Stock


#### Save Dataset

In [179]:
books_df.to_csv(

    f"{PROJECT_PATH}/data/books.csv",

    index=False

)

print("books.csv saved successfully!")

books.csv saved successfully!


#### Verify Dataset

In [180]:
print(f"{len(books_df)} books generated successfully!")

books_df.head()

268 books generated successfully!


,Book_ID,Subject_ID,Subject_Name,Book_Title,Author,Publisher,ISBN,Edition,Published_Year,Total_Copies,Available_Copies,Shelf_No,Status
0,BK001,SUB001,Mathematics-I,Mathematics-I Essentials,Christopher Raj,McGraw Hill,978-1-09-764367-7,5 Edition,2017,3,0,S-15-4,Out of Stock
1,BK002,SUB001,Mathematics-I,Advanced Mathematics-I,Yashawini Jhaveri,Packt,978-1-919993-32-4,5 Edition,2018,7,6,S-17-6,Available
2,BK003,SUB001,Mathematics-I,Mathematics-I Concepts,Neel Natt,Oxford University Press,978-1-997876-02-1,1 Edition,2025,3,1,S-03-5,Limited
3,BK004,SUB001,Mathematics-I,Introduction to Mathematics-I,Gautami Som,O'Reilly,978-1-70204-295-6,2 Edition,2022,7,6,S-13-4,Available
4,BK005,SUB002,Physics,Physics,Tripti Chand,Springer,978-1-78978-861-7,5 Edition,2016,8,0,S-06-7,Out of Stock


### J. Library Transactions Dataset

#### Configuration

In [181]:
MAX_BOOKS_PER_STUDENT = 3

BORROW_PROBABILITY = 0.40      # 40% students borrow books

LOAN_PERIOD_DAYS = 14

MAX_FINE = 500

#### Generate Transactions

In [182]:
library_transactions = []

transaction_counter = 1

today = datetime.today().date()

for _, student in students_df.iterrows():

    # Only some students borrow books
    if random.random() > BORROW_PROBABILITY:
        continue

    # Student borrows 1-3 unique books
    borrowed_books = books_df.sample(
        random.randint(1, MAX_BOOKS_PER_STUDENT)
    )

    for _, book in borrowed_books.iterrows():

        issue_date = fake.date_between(
            start_date="-180d",
            end_date="-20d"
        )

        due_date = issue_date + timedelta(days=LOAN_PERIOD_DAYS)

        # 80% chance the book has been returned
        if random.random() < 0.8:

            return_date = fake.date_between(
                start_date=issue_date,
                end_date=today
            )

            delay = max(
                (return_date - due_date).days,
                0
            )

            fine = min(delay * 10, MAX_FINE)

            status = "Returned"

        else:

            return_date = None

            if today > due_date:

                delay = (today - due_date).days

                fine = min(delay * 10, MAX_FINE)

                status = "Overdue"

            else:

                fine = 0

                status = "Issued"

        library_transactions.append({

            "Transaction_ID":
                generate_id("LIB", transaction_counter),

            "Student_ID":
                student["Student_ID"],

            "Student_Name":
                student["Full_Name"],

            "Book_ID":
                book["Book_ID"],

            "Book_Title":
                book["Book_Title"],

            "Issue_Date":
                issue_date,

            "Due_Date":
                due_date,

            "Return_Date":
                return_date,

            "Fine_Amount":
                fine,

            "Status":
                status

        })

        transaction_counter += 1

library_transactions_df = pd.DataFrame(
    library_transactions
)

library_transactions_df

,Transaction_ID,Student_ID,Student_Name,Book_ID,Book_Title,Issue_Date,Due_Date,Return_Date,Fine_Amount,Status
0,LIB001,STU0004,Harshil Sethi,BK256,Fundamentals of Business Analytics,2026-02-16,2026-03-02,2026-07-08,500,Returned
1,LIB002,STU0004,Harshil Sethi,BK028,Data Structures Concepts,2026-03-07,2026-03-21,2026-03-25,40,Returned
2,LIB003,STU0005,Rishi Ben,BK047,Advanced Operating Systems,2026-02-27,2026-03-13,2026-04-01,190,Returned
3,LIB004,STU0006,Mahika Dar,BK057,Advanced Java Programming,2026-01-31,2026-02-14,None,500,Overdue
4,LIB005,STU0010,Chakrika Wali,BK213,Advanced Business Analytics,2026-03-04,2026-03-18,2026-04-04,170,Returned
...,...,...,...,...,...,...,...,...,...,...
2364,LIB2365,STU2989,Mason Sundaram,BK244,Financial Management,2026-04-04,2026-04-18,2026-05-04,160,Returned
2365,LIB2366,STU2989,Mason Sundaram,BK001,Mathematics-I Essentials,2026-01-28,2026-02-11,2026-03-29,460,Returned
2366,LIB2367,STU3000,Dhruv Gupta,BK154,Introduction to Organizational Behaviour,2026-05-05,2026-05-19,None,500,Overdue
2367,LIB2368,STU3000,Dhruv Gupta,BK177,Introduction to Operations Management,2026-04-02,2026-04-16,2026-05-12,260,Returned


#### Save Dataset

In [183]:
library_transactions_df.to_csv(
    f"{PROJECT_PATH}/data/library_transactions.csv",
    index=False
)

print("library_transactions.csv saved successfully!")

library_transactions.csv saved successfully!


#### Verify Dataset


In [184]:
print("Total Transactions:", len(library_transactions_df))

library_transactions_df.head()

Total Transactions: 2369


,Transaction_ID,Student_ID,Student_Name,Book_ID,Book_Title,Issue_Date,Due_Date,Return_Date,Fine_Amount,Status
0,LIB001,STU0004,Harshil Sethi,BK256,Fundamentals of Business Analytics,2026-02-16,2026-03-02,2026-07-08,500,Returned
1,LIB002,STU0004,Harshil Sethi,BK028,Data Structures Concepts,2026-03-07,2026-03-21,2026-03-25,40,Returned
2,LIB003,STU0005,Rishi Ben,BK047,Advanced Operating Systems,2026-02-27,2026-03-13,2026-04-01,190,Returned
3,LIB004,STU0006,Mahika Dar,BK057,Advanced Java Programming,2026-01-31,2026-02-14,None,500,Overdue
4,LIB005,STU0010,Chakrika Wali,BK213,Advanced Business Analytics,2026-03-04,2026-03-18,2026-04-04,170,Returned


### K. Attendance Dataset
    

#### Configuration

In [185]:
ATTENDANCE_DAYS = 30          # Generate attendance for last 30 working days

TOTAL_CLASSES_PER_COURSE = 45

ATTENDANCE_RANGE = (75, 98)

#### Generate Dataset

In [186]:
attendance = []

attendance_counter = 1

for _, student in students_df.iterrows():

    student_courses = courses_df[
        courses_df["Section_ID"] == student["Section_ID"]
    ]

    # Give each student a personal attendance percentage
    attendance_percentage = random.randint(
        ATTENDANCE_RANGE[0],
        ATTENDANCE_RANGE[1]
    )

    for _, course in student_courses.iterrows():

        classes_conducted = TOTAL_CLASSES_PER_COURSE

        classes_attended = round(
            classes_conducted *
            attendance_percentage / 100
        )

        attendance.append({

            "Attendance_ID":
                generate_id(
                    "ATT",
                    attendance_counter
                ),

            "Student_ID":
                student["Student_ID"],

            "Course_ID":
                course["Course_ID"],

            "Classes_Conducted":
                classes_conducted,

            "Classes_Attended":
                classes_attended,

            "Attendance_Percentage":
                round(
                    classes_attended /
                    classes_conducted * 100,
                    2
                )

        })

        attendance_counter += 1

attendance_df = pd.DataFrame(attendance)

attendance_df

,Attendance_ID,Student_ID,Course_ID,Classes_Conducted,Classes_Attended,Attendance_Percentage
0,ATT001,STU0001,CRS001,45,35,77.78
1,ATT002,STU0001,CRS002,45,35,77.78
2,ATT003,STU0001,CRS003,45,35,77.78
3,ATT004,STU0001,CRS004,45,35,77.78
4,ATT005,STU0001,CRS005,45,35,77.78
...,...,...,...,...,...,...
29995,ATT29996,STU3000,CRS496,45,37,82.22
29996,ATT29997,STU3000,CRS497,45,37,82.22
29997,ATT29998,STU3000,CRS498,45,37,82.22
29998,ATT29999,STU3000,CRS499,45,37,82.22


#### Save Dataset

In [187]:
attendance_df.to_csv(
    f"{PROJECT_PATH}/data/attendance.csv",
    index=False
)

print("attendance.csv saved successfully!")

attendance.csv saved successfully!


#### Verify Dataset

In [188]:
print("Total Records:", len(attendance_df))

attendance_df.head()

Total Records: 30000


,Attendance_ID,Student_ID,Course_ID,Classes_Conducted,Classes_Attended,Attendance_Percentage
0,ATT001,STU0001,CRS001,45,35,77.78
1,ATT002,STU0001,CRS002,45,35,77.78
2,ATT003,STU0001,CRS003,45,35,77.78
3,ATT004,STU0001,CRS004,45,35,77.78
4,ATT005,STU0001,CRS005,45,35,77.78


### L. Exam Dataset

#### Configuration

In [189]:
EXAM_TYPES = [
    "Mid Semester",
    "End Semester"
]

EXAM_START_DATE = datetime(2026, 11, 10)

EXAM_START_TIMES = [
    "09:00 AM",
    "02:00 PM"
]

#### Generate Dataset

In [190]:
exam_schedule = []

exam_counter = 1

exam_date = EXAM_START_DATE

for _, subject in subjects_df.iterrows():

    # Skip non-examination subjects
    if subject["Subject_Name"] in NO_BOOK_SUBJECTS:
        continue

    # Find all courses teaching this subject
    subject_courses = courses_df[
        courses_df["Subject_ID"] == subject["Subject_ID"]
    ]

    for exam_type in EXAM_TYPES:

        exam_schedule.append({

            "Exam_ID":
                generate_id("EXM", exam_counter),

            "Subject_ID":
                subject["Subject_ID"],

            "Course_ID":
                random.choice(subject_courses["Course_ID"].tolist()),

            "Exam_Type":
                exam_type,

            "Exam_Date":
                exam_date.date(),

            "Exam_Time":
                random.choice(EXAM_START_TIMES),

            "Duration_Hours":
                3,

            "Room_ID":
                random.choice(classrooms_df["Room_ID"].tolist())

        })

        exam_counter += 1

    # Next subject exam after one day
    exam_date += timedelta(days=1)

exam_schedule_df = pd.DataFrame(exam_schedule)

exam_schedule_df

,Exam_ID,Subject_ID,Course_ID,Exam_Type,Exam_Date,Exam_Time,Duration_Hours,Room_ID
0,EXM001,SUB001,CRS121,Mid Semester,2026-11-10,02:00 PM,3,ROOM043
1,EXM002,SUB001,CRS181,End Semester,2026-11-10,02:00 PM,3,ROOM001
2,EXM003,SUB002,CRS182,Mid Semester,2026-11-11,02:00 PM,3,ROOM034
3,EXM004,SUB002,CRS022,End Semester,2026-11-11,02:00 PM,3,ROOM018
4,EXM005,SUB003,CRS283,Mid Semester,2026-11-12,09:00 AM,3,ROOM037
...,...,...,...,...,...,...,...,...
129,EXM130,SUB080,CRS470,End Semester,2027-01-13,02:00 PM,3,ROOM026
130,EXM131,SUB081,CRS481,Mid Semester,2027-01-14,09:00 AM,3,ROOM016
131,EXM132,SUB081,CRS481,End Semester,2027-01-14,09:00 AM,3,ROOM003
132,EXM133,SUB082,CRS482,Mid Semester,2027-01-15,02:00 PM,3,ROOM009


#### Save Dataset

In [191]:
exam_schedule_df.to_csv(
    f"{PROJECT_PATH}/data/exam_schedule.csv",
    index=False
)

print("exam_schedule.csv saved successfully!")


exam_schedule.csv saved successfully!


#### Verify Dataset

In [192]:
print("Shape:", exam_schedule_df.shape)

exam_schedule_df.head()

exam_schedule_df.info()

Shape: (134, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Exam_ID         134 non-null    object
 1   Subject_ID      134 non-null    object
 2   Course_ID       134 non-null    object
 3   Exam_Type       134 non-null    object
 4   Exam_Date       134 non-null    object
 5   Exam_Time       134 non-null    object
 6   Duration_Hours  134 non-null    int64 
 7   Room_ID         134 non-null    object
dtypes: int64(1), object(7)
memory usage: 8.5+ KB


### M. Marks Dataset

#### Configuration

In [193]:
MAX_ASSIGNMENT = 10

MAX_QUIZ = 10

MAX_MIDSEM = 30

MAX_ENDSEM = 50

#### Helper Function

In [194]:
def calculate_grade(total):

    if total >= 90:
        return "A+", 10

    elif total >= 80:
        return "A", 9

    elif total >= 70:
        return "B+", 8

    elif total >= 60:
        return "B", 7

    elif total >= 50:
        return "C", 6

    elif total >= 40:
        return "D", 5

    else:
        return "F", 0

#### Generate Dataset

In [195]:
marks = []

marks_counter = 1

for _, student in students_df.iterrows():

    student_courses = courses_df[
        courses_df["Section_ID"] == student["Section_ID"]
    ]

    for _, course in student_courses.iterrows():

        assignment = random.randint(5, MAX_ASSIGNMENT)

        quiz = random.randint(4, MAX_QUIZ)

        midsem = random.randint(10, MAX_MIDSEM)

        endsem = random.randint(18, MAX_ENDSEM)

        total = (
            assignment +
            quiz +
            midsem +
            endsem
        )

        grade, grade_point = calculate_grade(total)

        marks.append({

            "Mark_ID":
                generate_id(
                    "MRK",
                    marks_counter
                ),

            "Student_ID":
                student["Student_ID"],

            "Course_ID":
                course["Course_ID"],

            "Assignment":
                assignment,

            "Quiz":
                quiz,

            "Mid_Sem":
                midsem,

            "End_Sem":
                endsem,

            "Total":
                total,

            "Grade":
                grade,

            "Grade_Point":
                grade_point

        })

        marks_counter += 1

marks_df = pd.DataFrame(marks)

marks_df

,Mark_ID,Student_ID,Course_ID,Assignment,Quiz,Mid_Sem,End_Sem,Total,Grade,Grade_Point
0,MRK001,STU0001,CRS001,5,6,23,28,62,B,7
1,MRK002,STU0001,CRS002,5,9,25,49,88,A,9
2,MRK003,STU0001,CRS003,7,8,18,26,59,C,6
3,MRK004,STU0001,CRS004,9,6,16,45,76,B+,8
4,MRK005,STU0001,CRS005,7,5,11,25,48,D,5
...,...,...,...,...,...,...,...,...,...,...
29995,MRK29996,STU3000,CRS496,8,5,12,33,58,C,6
29996,MRK29997,STU3000,CRS497,8,9,11,33,61,B,7
29997,MRK29998,STU3000,CRS498,7,6,26,23,62,B,7
29998,MRK29999,STU3000,CRS499,8,9,22,43,82,A,9


#### Save Dataset

In [196]:
marks_df.to_csv(
    f"{PROJECT_PATH}/data/marks.csv",
    index=False
)

print("marks.csv saved successfully!")



marks.csv saved successfully!


#### Verify Dataset


In [197]:
print("Total Records:", len(marks_df))

marks_df.head()

Total Records: 30000


,Mark_ID,Student_ID,Course_ID,Assignment,Quiz,Mid_Sem,End_Sem,Total,Grade,Grade_Point
0,MRK001,STU0001,CRS001,5,6,23,28,62,B,7
1,MRK002,STU0001,CRS002,5,9,25,49,88,A,9
2,MRK003,STU0001,CRS003,7,8,18,26,59,C,6
3,MRK004,STU0001,CRS004,9,6,16,45,76,B+,8
4,MRK005,STU0001,CRS005,7,5,11,25,48,D,5


### N. Fees Dataset

#### Configuration

In [198]:
SEMESTERS_PER_PROGRAM = {
    "B.Tech": 8,
    "BBA": 6,
    "MBA": 4
}

TUITION_FEE = {
    "B.Tech": 90000,
    "BBA": 60000,
    "MBA": 100000
}

LIBRARY_FEE = 3000

EXAM_FEE = 2500

#### Generate Dataset

In [199]:
fees = []

fee_counter = 1

for _, student in students_df.iterrows():

    total_semesters = SEMESTERS_PER_PROGRAM[
        student["Program"]
    ]

    for semester in range(1, total_semesters + 1):

        tuition = TUITION_FEE[
            student["Program"]
        ]

        total_fee = (
            tuition +
            LIBRARY_FEE +
            EXAM_FEE
        )

        payment_type = random.choices(

            ["Paid", "Partially Paid", "Pending"],

            weights=[65, 25, 10],

            k=1

        )[0]

        if payment_type == "Paid":

            amount_paid = total_fee

        elif payment_type == "Partially Paid":

            amount_paid = random.randint(
                total_fee // 2,
                total_fee - 1000
            )

        else:

            amount_paid = 0

        balance = total_fee - amount_paid

        fees.append({

            "Fee_ID":
                generate_id(
                    "FEE",
                    fee_counter
                ),

            "Student_ID":
                student["Student_ID"],

            "Semester":
                semester,

            "Tuition_Fee":
                tuition,

            "Library_Fee":
                LIBRARY_FEE,

            "Exam_Fee":
                EXAM_FEE,

            "Total_Fee":
                total_fee,

            "Amount_Paid":
                amount_paid,

            "Balance":
                balance,

            "Status":
                payment_type

        })

        fee_counter += 1

fees_df = pd.DataFrame(fees)

fees_df

,Fee_ID,Student_ID,Semester,Tuition_Fee,Library_Fee,Exam_Fee,Total_Fee,Amount_Paid,Balance,Status
0,FEE001,STU0001,1,90000,3000,2500,95500,95500,0,Paid
1,FEE002,STU0001,2,90000,3000,2500,95500,95500,0,Paid
2,FEE003,STU0001,3,90000,3000,2500,95500,95500,0,Paid
3,FEE004,STU0001,4,90000,3000,2500,95500,95500,0,Paid
4,FEE005,STU0001,5,90000,3000,2500,95500,95500,0,Paid
...,...,...,...,...,...,...,...,...,...,...
22315,FEE22316,STU2999,4,100000,3000,2500,105500,105500,0,Paid
22316,FEE22317,STU3000,1,100000,3000,2500,105500,105500,0,Paid
22317,FEE22318,STU3000,2,100000,3000,2500,105500,105500,0,Paid
22318,FEE22319,STU3000,3,100000,3000,2500,105500,105500,0,Paid


#### Save Dataset

In [200]:
fees_df.to_csv(

    f"{PROJECT_PATH}/data/fees.csv",

    index=False

)

print("fees.csv saved successfully!")

fees.csv saved successfully!


#### Verify Dataset

In [201]:
print("Total Records:", len(fees_df))

fees_df.head()

Total Records: 22320


,Fee_ID,Student_ID,Semester,Tuition_Fee,Library_Fee,Exam_Fee,Total_Fee,Amount_Paid,Balance,Status
0,FEE001,STU0001,1,90000,3000,2500,95500,95500,0,Paid
1,FEE002,STU0001,2,90000,3000,2500,95500,95500,0,Paid
2,FEE003,STU0001,3,90000,3000,2500,95500,95500,0,Paid
3,FEE004,STU0001,4,90000,3000,2500,95500,95500,0,Paid
4,FEE005,STU0001,5,90000,3000,2500,95500,95500,0,Paid


# ✅ Phase 1 Complete

All synthetic College ERP datasets have been generated successfully.

Generated datasets:

- Departments
- Faculty
- Sections
- Students
- Subjects
- Courses
- Classrooms
- Timetable
- Books
- Library Transactions
- Attendance
- Exam Schedule
- Marks
- Fees

The next phase of the CampusPilot project will be implemented in VS Code and includes:

- Data preprocessing
- RAG pipeline
- Vector database
- Agentic AI
- Streamlit UI